In [21]:
import os
import pandas as pd
import numpy as np
from glob import glob
from scipy.spatial.distance import jensenshannon
import matplotlib as mpl

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === CONFIGURATION ===
proportions_file = 'TOO-Decon/20250616_All-Tissues-NoDup_Random_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon'
results_dir = os.path.join(root_dir, 'Results-JSD-PredictedVSActual')
os.makedirs(results_dir, exist_ok=True)

method_map = ['BayesPrism', 'nuSVR', 'ReDeconv', 'CIBERSORTx', 'MuSiC', 'NNLS', 'QP']

# === LOAD PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)

prop_df.index = prop_df.index.astype(str).str.strip()
prop_df.columns = prop_df.columns.astype(str).str.strip()

prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)

prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})

prop_df = prop_df.drop(
    columns=[
        'Cervix', 'Ovary', 'Uterus',
        'Thyroid-gland', 'Adrenal-gland',
        'Small-Intestine', 'Skeletal-muscle'
    ],
    errors='ignore'
)

reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast',
    'Colon', 'Esophagus', 'FemaleReproductive', 'Fibroblasts', 'Heart',
    'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal',
    'NerveTibial', 'Pancreas', 'Pituitary', 'Prostate', 'SalivaryGland',
    'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid',
    'Whole blood'
]

# Add missing tissue columns
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0

prop_df = prop_df[reference_tissues]
prop_df = prop_df.apply(pd.to_numeric, errors='coerce').fillna(0)

# Normalize truth per sample
prop_sums = prop_df.sum(axis=1)

if (prop_sums == 0).any():
    print(f"Warning: {(prop_sums == 0).sum()} true-proportion rows sum to zero.")

prop_df = (
    prop_df
    .div(prop_sums.replace(0, np.nan), axis=0)
    .multiply(100)
    .round(5)
    .fillna(0)
)

# === PROCESS EACH DECON FILE ===
decon_files = sorted(glob(os.path.join(root_dir, 'Decon-Results_*Random_TOOv2_1000', '*_modified.txt')))

results = []

for file_path in decon_files:
    try:
        dir_name = os.path.basename(os.path.dirname(file_path))
        matrix = dir_name.replace('Decon-Results_', '').split('-')[0]

        filename = os.path.basename(file_path)
        raw_method = filename.replace('_modified.txt', '')
        method = next((m for m in method_map if m in raw_method), raw_method)

        decon_df = pd.read_csv(file_path, sep='\t', index_col=0)

        decon_df.index = decon_df.index.astype(str).str.strip()
        decon_df.columns = decon_df.columns.astype(str).str.strip()

        for col in reference_tissues:
            if col not in decon_df.columns:
                decon_df[col] = 0.0

        decon_df = decon_df[reference_tissues]
        decon_df = decon_df.apply(pd.to_numeric, errors='coerce').fillna(0)

        common_samples = prop_df.index.intersection(decon_df.index)

        if common_samples.empty:
            print(f"No common samples in {file_path}")
            continue

        true_vals_df = prop_df.loc[common_samples, reference_tissues].fillna(0)
        pred_vals_df = decon_df.loc[common_samples, reference_tissues].fillna(0)

        sample_jsd = []
        skipped_samples = 0

        for sample in common_samples:
            true_vec = true_vals_df.loc[sample].values.astype(float)
            pred_vec = pred_vals_df.loc[sample].values.astype(float)

            true_vec = np.clip(true_vec, 0, None)
            pred_vec = np.clip(pred_vec, 0, None)

            true_sum = true_vec.sum()
            pred_sum = pred_vec.sum()

            if true_sum == 0 or pred_sum == 0:
                skipped_samples += 1
                continue

            true_vec = true_vec / true_sum
            pred_vec = pred_vec / pred_sum

            js_distance = jensenshannon(true_vec, pred_vec, base=2)
            js_divergence = js_distance ** 2

            sample_jsd.append(js_divergence)

        n_samples = len(sample_jsd)

        if n_samples == 0:
            print(f"No valid samples for {matrix} — {method}")
            continue

        mean_jsd = np.mean(sample_jsd)
        median_jsd = np.median(sample_jsd)
        std_jsd = np.std(sample_jsd)
        min_jsd = np.min(sample_jsd)
        max_jsd = np.max(sample_jsd)

        print(
            f"{matrix} — {method}: "
            f"N={n_samples}, Skipped={skipped_samples}, "
            f"Mean JSD={mean_jsd:.6f}, Median JSD={median_jsd:.6f}, SD={std_jsd:.6f}"
        )

        results.append([
            matrix,
            method,
            mean_jsd,
            median_jsd,
            std_jsd,
            min_jsd,
            max_jsd,
            n_samples,
            skipped_samples
        ])

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# === SAVE SUMMARY TABLE ===
if results:
    results_df = pd.DataFrame(
        results,
        columns=[
            "Matrix",
            "Method",
            "Mean_JS_divergence",
            "Median_JS_divergence",
            "Std_JS_divergence",
            "Min_JS_divergence",
            "Max_JS_divergence",
            "N_samples",
            "Skipped_samples"
        ]
    )

    results_csv = os.path.join(results_dir, "Random_JSD_Summary.csv")
    results_df.to_csv(results_csv, index=False)

    print(f"Saved JSD summary table: {results_csv}")

else:
    print("No JSD results to save.")

2Median_1000_1500 — ReDeconv: N=1000, Skipped=0, Mean JSD=0.839532, Median JSD=0.897233, SD=0.169646
2Median_1000_1500 — BayesPrism: N=1000, Skipped=0, Mean JSD=0.400167, Median JSD=0.385341, SD=0.163589
2Median_1000_1500 — MuSiC: N=1000, Skipped=0, Mean JSD=0.643157, Median JSD=0.669131, SD=0.185834
2Median_1000_1500 — CIBERSORTx: N=1000, Skipped=0, Mean JSD=0.488644, Median JSD=0.490836, SD=0.156714
2Median_1000_1500 — NNLS: N=1000, Skipped=0, Mean JSD=0.582228, Median JSD=0.583719, SD=0.174504
2Median_1000_1500 — QP: N=1000, Skipped=0, Mean JSD=0.578995, Median JSD=0.584338, SD=0.163620
2Median_1000_1500 — nuSVR: N=1000, Skipped=0, Mean JSD=0.419707, Median JSD=0.399270, SD=0.142729
2Median_300_500 — ReDeconv: N=1000, Skipped=0, Mean JSD=0.838036, Median JSD=0.901107, SD=0.176986
2Median_300_500 — BayesPrism: N=1000, Skipped=0, Mean JSD=0.400167, Median JSD=0.385341, SD=0.163589
2Median_300_500 — MuSiC: N=1000, Skipped=0, Mean JSD=0.643157, Median JSD=0.669131, SD=0.185834
2Median_3

In [22]:
import os
import pandas as pd
import numpy as np
from glob import glob
from scipy.spatial.distance import jensenshannon
import matplotlib as mpl

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === CONFIGURATION ===
proportions_file = 'TOO-Decon/20250616_All-Tissues-NoDup_Uniform_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon'
results_dir = os.path.join(root_dir, 'Results-JSD-PredictedVSActual')
os.makedirs(results_dir, exist_ok=True)

method_map = ['BayesPrism', 'nuSVR', 'ReDeconv', 'CIBERSORTx', 'MuSiC', 'NNLS', 'QP']

# === LOAD PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)

prop_df.index = prop_df.index.astype(str).str.strip()
prop_df.columns = prop_df.columns.astype(str).str.strip()

prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)

prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})

prop_df = prop_df.drop(
    columns=[
        'Cervix', 'Ovary', 'Uterus',
        'Thyroid-gland', 'Adrenal-gland',
        'Small-Intestine', 'Skeletal-muscle'
    ],
    errors='ignore'
)

reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast',
    'Colon', 'Esophagus', 'FemaleReproductive', 'Fibroblasts', 'Heart',
    'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal',
    'NerveTibial', 'Pancreas', 'Pituitary', 'Prostate', 'SalivaryGland',
    'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid',
    'Whole blood'
]

# Add missing tissue columns
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0

prop_df = prop_df[reference_tissues]
prop_df = prop_df.apply(pd.to_numeric, errors='coerce').fillna(0)

# Normalize truth per sample
prop_sums = prop_df.sum(axis=1)

if (prop_sums == 0).any():
    print(f"Warning: {(prop_sums == 0).sum()} true-proportion rows sum to zero.")

prop_df = (
    prop_df
    .div(prop_sums.replace(0, np.nan), axis=0)
    .multiply(100)
    .round(5)
    .fillna(0)
)

# === PROCESS EACH DECON FILE ===
decon_files = sorted(glob(os.path.join(root_dir, 'Decon-Results_*Uniform_TOOv2_250', '*_modified.txt')))

results = []

for file_path in decon_files:
    try:
        dir_name = os.path.basename(os.path.dirname(file_path))
        matrix = dir_name.replace('Decon-Results_', '').split('-')[0]

        filename = os.path.basename(file_path)
        raw_method = filename.replace('_modified.txt', '')
        method = next((m for m in method_map if m in raw_method), raw_method)

        decon_df = pd.read_csv(file_path, sep='\t', index_col=0)

        decon_df.index = decon_df.index.astype(str).str.strip()
        decon_df.columns = decon_df.columns.astype(str).str.strip()

        for col in reference_tissues:
            if col not in decon_df.columns:
                decon_df[col] = 0.0

        decon_df = decon_df[reference_tissues]
        decon_df = decon_df.apply(pd.to_numeric, errors='coerce').fillna(0)

        common_samples = prop_df.index.intersection(decon_df.index)

        if common_samples.empty:
            print(f"No common samples in {file_path}")
            continue

        true_vals_df = prop_df.loc[common_samples, reference_tissues].fillna(0)
        pred_vals_df = decon_df.loc[common_samples, reference_tissues].fillna(0)

        sample_jsd = []
        skipped_samples = 0

        for sample in common_samples:
            true_vec = true_vals_df.loc[sample].values.astype(float)
            pred_vec = pred_vals_df.loc[sample].values.astype(float)

            true_vec = np.clip(true_vec, 0, None)
            pred_vec = np.clip(pred_vec, 0, None)

            true_sum = true_vec.sum()
            pred_sum = pred_vec.sum()

            if true_sum == 0 or pred_sum == 0:
                skipped_samples += 1
                continue

            true_vec = true_vec / true_sum
            pred_vec = pred_vec / pred_sum

            js_distance = jensenshannon(true_vec, pred_vec, base=2)
            js_divergence = js_distance ** 2

            sample_jsd.append(js_divergence)

        n_samples = len(sample_jsd)

        if n_samples == 0:
            print(f"No valid samples for {matrix} — {method}")
            continue

        mean_jsd = np.mean(sample_jsd)
        median_jsd = np.median(sample_jsd)
        std_jsd = np.std(sample_jsd)
        min_jsd = np.min(sample_jsd)
        max_jsd = np.max(sample_jsd)

        print(
            f"{matrix} — {method}: "
            f"N={n_samples}, Skipped={skipped_samples}, "
            f"Mean JSD={mean_jsd:.6f}, Median JSD={median_jsd:.6f}, SD={std_jsd:.6f}"
        )

        results.append([
            matrix,
            method,
            mean_jsd,
            median_jsd,
            std_jsd,
            min_jsd,
            max_jsd,
            n_samples,
            skipped_samples
        ])

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# === SAVE SUMMARY TABLE ===
if results:
    results_df = pd.DataFrame(
        results,
        columns=[
            "Matrix",
            "Method",
            "Mean_JS_divergence",
            "Median_JS_divergence",
            "Std_JS_divergence",
            "Min_JS_divergence",
            "Max_JS_divergence",
            "N_samples",
            "Skipped_samples"
        ]
    )

    results_csv = os.path.join(results_dir, "Uniform_JSD_Summary.csv")
    results_df.to_csv(results_csv, index=False)

    print(f"Saved JSD summary table: {results_csv}")

else:
    print("No JSD results to save.")

2Median_1000_1500 — ReDeconv: N=250, Skipped=0, Mean JSD=0.826450, Median JSD=0.883985, SD=0.172957
2Median_1000_1500 — BayesPrism: N=250, Skipped=0, Mean JSD=0.417566, Median JSD=0.418912, SD=0.140881
2Median_1000_1500 — MuSiC: N=250, Skipped=0, Mean JSD=0.651430, Median JSD=0.655231, SD=0.155113
2Median_1000_1500 — CIBERSORTx: N=250, Skipped=0, Mean JSD=0.503073, Median JSD=0.509516, SD=0.137214
2Median_1000_1500 — NNLS: N=250, Skipped=0, Mean JSD=0.573256, Median JSD=0.579180, SD=0.148118
2Median_1000_1500 — QP: N=250, Skipped=0, Mean JSD=0.573405, Median JSD=0.587124, SD=0.143053
2Median_1000_1500 — nuSVR: N=250, Skipped=0, Mean JSD=0.426826, Median JSD=0.421902, SD=0.116832
2Median_300_500 — ReDeconv: N=250, Skipped=0, Mean JSD=0.825966, Median JSD=0.890118, SD=0.178117
2Median_300_500 — BayesPrism: N=250, Skipped=0, Mean JSD=0.417566, Median JSD=0.418912, SD=0.140881
2Median_300_500 — MuSiC: N=250, Skipped=0, Mean JSD=0.651430, Median JSD=0.655231, SD=0.155113
2Median_300_500 — C

In [23]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
import matplotlib as mpl

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
df = pd.read_csv("TOO-Decon/Results-JSD-PredictedVSActual/Random_JSD_Summary.csv")

# ------------------ Pivot for heatmap ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Mean_JS_divergence',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# ------------------ Build annotations ------------------
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))

annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig'] = idx_str.str.split('_').str[-1].astype(int)
annotations['Sampling'] = pd.Categorical(
    annotations['Sampling'],
    categories=sampling_order,
    ordered=True
)

# ------------------ Sort rows ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# ------------------ Recompute annotations after sorting ------------------
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))

annotations['Sampling'] = pd.Categorical(
    idx_str.str.split('_').str[0],
    categories=sampling_order,
    ordered=True
)
annotations['MaxSig'] = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
method_order = [c for c in method_order if c in heatmap_data.columns]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series = pd.Series(annotations['MaxSig'].astype(int).values, index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig': [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

vmin = heatmap_data.min().min()
vmax = heatmap_data.max().max()

norm = Normalize(vmin=vmin, vmax=vmax)

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='viridis_r',          # lower JSD = better
    norm=norm,
    figsize=(8, 5),
    annot=True,
    fmt=".2f",
    annot_kws={"size": 11, "weight": "normal"},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02)
)

# === Axis styling ===
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=10)
g.ax_heatmap.yaxis.set_label_position("left")

g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=14, labelpad=10)
g.ax_heatmap.tick_params(
    axis='x',
    bottom=True,
    labelbottom=True,
    length=4,
    width=0.6,
    labelsize=12
    )

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])
sm = plt.cm.ScalarMappable(cmap='viridis_r', norm=norm)
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)

cbar.ax.set_ylabel(
    'Mean Jensen–Shannon divergence',
    fontsize=13,
    rotation=270,
    labelpad=20
)

cbar.ax.tick_params(labelsize=11, width=0.8, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends ===
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

# Sampling legend
legend_ax1 = g.fig.add_axes([0.02, 0.60, 0.20, 0.20])
legend_ax1.axis('off')
legend1 = legend_ax1.legend(
    handles=sampling_patches,
    title='Sampling',
    loc='center',
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    facecolor='white',
    edgecolor='lightgrey'
)
legend1.get_frame().set_linewidth(0.8)

# Max Signatures legend (lower)
legend_ax2 = g.fig.add_axes([0.02, 0.37, 0.20, 0.20])
legend_ax2.axis('off')
legend2 = legend_ax2.legend(
    handles=maxsig_patches,
    title='Max Signatures',
    loc='center',
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    facecolor='white',
    edgecolor='lightgrey'
)
legend2.get_frame().set_linewidth(0.8)

g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.18, left=0.25, right=0.85)

g.fig.savefig("Heatmap_Random_Mean_JSD_Final.svg")
plt.close(g.fig)

In [24]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
import matplotlib as mpl

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
df = pd.read_csv("TOO-Decon/Results-JSD-PredictedVSActual/Random_JSD_Summary.csv")

# ------------------ Pivot for heatmap ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Mean_JS_divergence',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# ------------------ Build annotations ------------------
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))

annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig'] = idx_str.str.split('_').str[-1].astype(int)
annotations['Sampling'] = pd.Categorical(
    annotations['Sampling'],
    categories=sampling_order,
    ordered=True
)

# ------------------ Sort rows ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# ------------------ Recompute annotations after sorting ------------------
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))

annotations['Sampling'] = pd.Categorical(
    idx_str.str.split('_').str[0],
    categories=sampling_order,
    ordered=True
)
annotations['MaxSig'] = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
method_order = [c for c in method_order if c in heatmap_data.columns]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series = pd.Series(annotations['MaxSig'].astype(int).values, index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig': [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

vmin = heatmap_data.min().min()
vmax = heatmap_data.max().max()

norm = Normalize(vmin=vmin, vmax=vmax)

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='viridis_r',          # lower JSD = better
    norm=norm,
    figsize=(8, 5),
#    annot=True,
#    fmt=".2f",
#    annot_kws={"size": 11, "weight": "normal"},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02)
)

# === Axis styling ===
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=10)
g.ax_heatmap.yaxis.set_label_position("left")

g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=14, labelpad=10)
g.ax_heatmap.tick_params(
    axis='x',
    bottom=True,
    labelbottom=True,
    length=4,
    width=0.6,
    labelsize=12
    )

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])
sm = plt.cm.ScalarMappable(cmap='viridis_r', norm=norm)
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)

cbar.ax.set_ylabel(
    'Mean Jensen–Shannon divergence',
    fontsize=13,
    rotation=270,
    labelpad=20
)

cbar.ax.tick_params(labelsize=11, width=0.8, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends ===
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

# Sampling legend
legend_ax1 = g.fig.add_axes([0.02, 0.60, 0.20, 0.20])
legend_ax1.axis('off')
legend1 = legend_ax1.legend(
    handles=sampling_patches,
    title='Sampling',
    loc='center',
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    facecolor='white',
    edgecolor='lightgrey'
)
legend1.get_frame().set_linewidth(0.8)

# Max Signatures legend (lower)
legend_ax2 = g.fig.add_axes([0.02, 0.37, 0.20, 0.20])
legend_ax2.axis('off')
legend2 = legend_ax2.legend(
    handles=maxsig_patches,
    title='Max Signatures',
    loc='center',
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    facecolor='white',
    edgecolor='lightgrey'
)
legend2.get_frame().set_linewidth(0.8)

g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.18, left=0.25, right=0.85)

g.fig.savefig("Heatmap_Random_Mean_JSD_Final_Clean.svg")
plt.close(g.fig)

In [25]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
import matplotlib as mpl

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
df = pd.read_csv("TOO-Decon/Results-JSD-PredictedVSActual/Uniform_JSD_Summary.csv")

# ------------------ Pivot for heatmap ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Mean_JS_divergence',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# ------------------ Build annotations ------------------
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))

annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig'] = idx_str.str.split('_').str[-1].astype(int)
annotations['Sampling'] = pd.Categorical(
    annotations['Sampling'],
    categories=sampling_order,
    ordered=True
)

# ------------------ Sort rows ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# ------------------ Recompute annotations after sorting ------------------
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))

annotations['Sampling'] = pd.Categorical(
    idx_str.str.split('_').str[0],
    categories=sampling_order,
    ordered=True
)
annotations['MaxSig'] = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
method_order = [c for c in method_order if c in heatmap_data.columns]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series = pd.Series(annotations['MaxSig'].astype(int).values, index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig': [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

vmin = heatmap_data.min().min()
vmax = heatmap_data.max().max()

norm = Normalize(vmin=vmin, vmax=vmax)

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='viridis_r',          # lower JSD = better
    norm=norm,
    figsize=(8, 5),
    annot=True,
    fmt=".2f",
    annot_kws={"size": 11, "weight": "normal"},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02)
)

# === Axis styling ===
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=10)
g.ax_heatmap.yaxis.set_label_position("left")

g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=14, labelpad=10)
g.ax_heatmap.tick_params(
    axis='x',
    bottom=True,
    labelbottom=True,
    length=4,
    width=0.6,
    labelsize=12
    )

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])
sm = plt.cm.ScalarMappable(cmap='viridis_r', norm=norm)
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)

cbar.ax.set_ylabel(
    'Mean Jensen–Shannon divergence',
    fontsize=13,
    rotation=270,
    labelpad=20
)

cbar.ax.tick_params(labelsize=11, width=0.8, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends ===
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

# Sampling legend
legend_ax1 = g.fig.add_axes([0.02, 0.60, 0.20, 0.20])
legend_ax1.axis('off')
legend1 = legend_ax1.legend(
    handles=sampling_patches,
    title='Sampling',
    loc='center',
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    facecolor='white',
    edgecolor='lightgrey'
)
legend1.get_frame().set_linewidth(0.8)

# Max Signatures legend (lower)
legend_ax2 = g.fig.add_axes([0.02, 0.37, 0.20, 0.20])
legend_ax2.axis('off')
legend2 = legend_ax2.legend(
    handles=maxsig_patches,
    title='Max Signatures',
    loc='center',
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    facecolor='white',
    edgecolor='lightgrey'
)
legend2.get_frame().set_linewidth(0.8)

g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.18, left=0.25, right=0.85)

g.fig.savefig("Heatmap_Uniform_Mean_JSD_Final.svg")
plt.close(g.fig)

In [26]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
import matplotlib as mpl

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
df = pd.read_csv("TOO-Decon/Results-JSD-PredictedVSActual/Uniform_JSD_Summary.csv")

# ------------------ Pivot for heatmap ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Mean_JS_divergence',
    aggfunc='mean'
)

# ------------------ Flatten any MultiIndex ------------------
def flatten_index(idx):
    return idx.map(lambda t: "_".join(map(str, t))) if isinstance(idx, pd.MultiIndex) else idx

heatmap_data.index = flatten_index(heatmap_data.index)
heatmap_data.columns = flatten_index(heatmap_data.columns)

# ------------------ Desired Sampling order ------------------
sampling_order = ["2Median", "Sampling5", "Sampling10"]

# ------------------ Build annotations ------------------
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))

annotations['Sampling'] = idx_str.str.split('_').str[0]
annotations['MaxSig'] = idx_str.str.split('_').str[-1].astype(int)
annotations['Sampling'] = pd.Categorical(
    annotations['Sampling'],
    categories=sampling_order,
    ordered=True
)

# ------------------ Sort rows ------------------
heatmap_data = (
    heatmap_data
    .assign(Sampling=annotations['Sampling'], MaxSig=annotations['MaxSig'])
    .sort_values(['Sampling', 'MaxSig'])
    .drop(columns=['Sampling', 'MaxSig'])
)

# ------------------ Recompute annotations after sorting ------------------
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))

annotations['Sampling'] = pd.Categorical(
    idx_str.str.split('_').str[0],
    categories=sampling_order,
    ordered=True
)
annotations['MaxSig'] = idx_str.str.split('_').str[-1].astype(int)

# ------------------ Reorder columns ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
method_order = [c for c in method_order if c in heatmap_data.columns]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
sampling_palette = sns.color_palette("Set2", n_colors=len(sampling_order))
sampling_lut = dict(zip(sampling_order, sampling_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# ------------------ Row colors ------------------
sampling_series = pd.Series(annotations['Sampling'].astype(str).values, index=annotations.index)
maxsig_series = pd.Series(annotations['MaxSig'].astype(int).values, index=annotations.index)

row_colors = pd.DataFrame({
    'Sampling': [sampling_lut[s] for s in sampling_series],
    'MaxSig': [maxsig_lut[int(k)] for k in maxsig_series]
}, index=annotations.index)[['Sampling', 'MaxSig']]

vmin = heatmap_data.min().min()
vmax = heatmap_data.max().max()

norm = Normalize(vmin=vmin, vmax=vmax)

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='viridis_r',          # lower JSD = better
    norm=norm,
    figsize=(8, 5),
#    annot=True,
#    fmt=".2f",
#    annot_kws={"size": 11, "weight": "normal"},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02)
)

# === Axis styling ===
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=10)
g.ax_heatmap.yaxis.set_label_position("left")

g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=14, labelpad=10)
g.ax_heatmap.tick_params(
    axis='x',
    bottom=True,
    labelbottom=True,
    length=4,
    width=0.6,
    labelsize=12
    )

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])
sm = plt.cm.ScalarMappable(cmap='viridis_r', norm=norm)
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)

cbar.ax.set_ylabel(
    'Mean Jensen–Shannon divergence',
    fontsize=13,
    rotation=270,
    labelpad=20
)

cbar.ax.tick_params(labelsize=11, width=0.8, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends ===
sampling_patches = [Patch(facecolor=sampling_lut[s], label=s) for s in sampling_order]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

# Sampling legend
legend_ax1 = g.fig.add_axes([0.02, 0.60, 0.20, 0.20])
legend_ax1.axis('off')
legend1 = legend_ax1.legend(
    handles=sampling_patches,
    title='Sampling',
    loc='center',
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    facecolor='white',
    edgecolor='lightgrey'
)
legend1.get_frame().set_linewidth(0.8)

# Max Signatures legend (lower)
legend_ax2 = g.fig.add_axes([0.02, 0.37, 0.20, 0.20])
legend_ax2.axis('off')
legend2 = legend_ax2.legend(
    handles=maxsig_patches,
    title='Max Signatures',
    loc='center',
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    facecolor='white',
    edgecolor='lightgrey'
)
legend2.get_frame().set_linewidth(0.8)

g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.18, left=0.25, right=0.85)

g.fig.savefig("Heatmap_Uniform_Mean_JSD_Final_Clean.svg")
plt.close(g.fig)

## DECODE extension

The cells below repeat the preceding calculation and plotting workflow with `DECODE-PureTrue` and `DECODE-PureFalse` included as additional methods.
The metric definitions and calculations are unchanged.

In [ ]:
import os
import pandas as pd
import numpy as np
from glob import glob
from scipy.spatial.distance import jensenshannon
import matplotlib as mpl

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === CONFIGURATION ===
proportions_file = 'TOO-Decon/20250616_All-Tissues-NoDup_Random_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon'
results_dir = os.path.join(root_dir, 'Results-JSD-PredictedVSActual')
os.makedirs(results_dir, exist_ok=True)

method_map = ['BayesPrism', 'nuSVR', 'ReDeconv', 'CIBERSORTx', 'MuSiC', 'NNLS', 'QP', 'DECODEPureTrue', 'DECODEPureFalse']

# === LOAD PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)

prop_df.index = prop_df.index.astype(str).str.strip()
prop_df.columns = prop_df.columns.astype(str).str.strip()

prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)

prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})

prop_df = prop_df.drop(
    columns=[
        'Cervix', 'Ovary', 'Uterus',
        'Thyroid-gland', 'Adrenal-gland',
        'Small-Intestine', 'Skeletal-muscle'
    ],
    errors='ignore'
)

reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast',
    'Colon', 'Esophagus', 'FemaleReproductive', 'Fibroblasts', 'Heart',
    'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal',
    'NerveTibial', 'Pancreas', 'Pituitary', 'Prostate', 'SalivaryGland',
    'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid',
    'Whole blood'
]

# Add missing tissue columns
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0

prop_df = prop_df[reference_tissues]
prop_df = prop_df.apply(pd.to_numeric, errors='coerce').fillna(0)

# Normalize truth per sample
prop_sums = prop_df.sum(axis=1)

if (prop_sums == 0).any():
    print(f"Warning: {(prop_sums == 0).sum()} true-proportion rows sum to zero.")

prop_df = (
    prop_df
    .div(prop_sums.replace(0, np.nan), axis=0)
    .multiply(100)
    .round(5)
    .fillna(0)
)

# === PROCESS EACH DECON FILE ===
decon_files = sorted(glob(os.path.join(root_dir, 'Decon-Results_*Random_TOOv2_1000', '*_modified.txt')))

results = []

for file_path in decon_files:
    try:
        dir_name = os.path.basename(os.path.dirname(file_path))
        matrix = dir_name.replace('Decon-Results_', '').split('-')[0]

        filename = os.path.basename(file_path)
        raw_method = filename.replace('_modified.txt', '')
        method = next((m for m in method_map if m in raw_method), raw_method)

        decon_df = pd.read_csv(file_path, sep='\t', index_col=0)

        decon_df.index = decon_df.index.astype(str).str.strip()
        decon_df.columns = decon_df.columns.astype(str).str.strip()

        for col in reference_tissues:
            if col not in decon_df.columns:
                decon_df[col] = 0.0

        decon_df = decon_df[reference_tissues]
        decon_df = decon_df.apply(pd.to_numeric, errors='coerce').fillna(0)

        common_samples = prop_df.index.intersection(decon_df.index)

        if common_samples.empty:
            print(f"No common samples in {file_path}")
            continue

        true_vals_df = prop_df.loc[common_samples, reference_tissues].fillna(0)
        pred_vals_df = decon_df.loc[common_samples, reference_tissues].fillna(0)

        sample_jsd = []
        skipped_samples = 0

        for sample in common_samples:
            true_vec = true_vals_df.loc[sample].values.astype(float)
            pred_vec = pred_vals_df.loc[sample].values.astype(float)

            true_vec = np.clip(true_vec, 0, None)
            pred_vec = np.clip(pred_vec, 0, None)

            true_sum = true_vec.sum()
            pred_sum = pred_vec.sum()

            if true_sum == 0 or pred_sum == 0:
                skipped_samples += 1
                continue

            true_vec = true_vec / true_sum
            pred_vec = pred_vec / pred_sum

            js_distance = jensenshannon(true_vec, pred_vec, base=2)
            js_divergence = js_distance ** 2

            sample_jsd.append(js_divergence)

        n_samples = len(sample_jsd)

        if n_samples == 0:
            print(f"No valid samples for {matrix} — {method}")
            continue

        mean_jsd = np.mean(sample_jsd)
        median_jsd = np.median(sample_jsd)
        std_jsd = np.std(sample_jsd)
        min_jsd = np.min(sample_jsd)
        max_jsd = np.max(sample_jsd)

        print(
            f"{matrix} — {method}: "
            f"N={n_samples}, Skipped={skipped_samples}, "
            f"Mean JSD={mean_jsd:.6f}, Median JSD={median_jsd:.6f}, SD={std_jsd:.6f}"
        )

        results.append([
            matrix,
            method,
            mean_jsd,
            median_jsd,
            std_jsd,
            min_jsd,
            max_jsd,
            n_samples,
            skipped_samples
        ])

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# === SAVE SUMMARY TABLE ===
if results:
    results_df = pd.DataFrame(
        results,
        columns=[
            "Matrix",
            "Method",
            "Mean_JS_divergence",
            "Median_JS_divergence",
            "Std_JS_divergence",
            "Min_JS_divergence",
            "Max_JS_divergence",
            "N_samples",
            "Skipped_samples"
        ]
    )

    results_csv = os.path.join(results_dir, "Random_JSD_Summary_DECODE.csv")
    results_df.to_csv(results_csv, index=False)

    print(f"Saved JSD summary table: {results_csv}")

else:
    print("No JSD results to save.")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load data ------------------
file_path = 'TOO-Decon/Results-JSD-PredictedVSActual/Random_JSD_Summary_DECODE.csv'
df = pd.read_csv(file_path)

jsd_column = 'Mean_JS_divergence'

# ------------------ Clean data ------------------
df[jsd_column] = pd.to_numeric(
    df[jsd_column],
    errors='coerce'
)

df = df.dropna(
    subset=['Method', 'Matrix', jsd_column]
)

# ------------------ Tool order ------------------
method_order = [
    'BayesPrism',
    'MuSiC',
    'nuSVR',
    'CIBERSORTx',
    'NNLS',
    'QP',
    'ReDeconv',
    'DECODEPureTrue'
]

# ------------------ Check for duplicate Method–Matrix rows ------------------
duplicates = df.duplicated(
    subset=['Method', 'Matrix'],
    keep=False
)

if duplicates.any():
    print(
        '\nWarning: duplicate Method–Matrix combinations found. '
        'Their Mean JSD values will be averaged.\n'
    )

    matrix_values = (
        df.groupby(
            ['Method', 'Matrix'],
            as_index=False
        )[jsd_column]
        .mean()
    )
else:
    matrix_values = df[
        ['Method', 'Matrix', jsd_column]
    ].copy()

# ------------------ Select optimal matrix per method ------------------
# Lower Mean JSD is better.
best_rows = (
    matrix_values
    .sort_values(
        ['Method', jsd_column, 'Matrix'],
        ascending=[True, True, True]
    )
    .drop_duplicates(
        subset='Method',
        keep='first'
    )
)

# ------------------ Print selected matrices ------------------
selected_results = (
    best_rows[
        ['Method', 'Matrix', jsd_column]
    ]
    .set_index('Method')
    .reindex(method_order)
    .reset_index()
)

print('\nSelected optimal matrix for each method:\n')

print(
    selected_results.to_string(
        index=False,
        formatters={
            jsd_column: lambda x: (
                f'{x:.6f}' if pd.notna(x) else 'NA'
            )
        }
    )
)

# Optional: print all matrices from lowest to highest Mean JSD
print('\nAll Mean JSD values for each method:\n')

print(
    matrix_values
    .sort_values(
        ['Method', jsd_column],
        ascending=[True, True]
    )
    .to_string(
        index=False,
        formatters={
            jsd_column: lambda x: f'{x:.6f}'
        }
    )
)

# ------------------ Prepare one-row heatmap ------------------
heatmap_data = (
    best_rows
    .set_index('Method')[[jsd_column]]
    .reindex(method_order)
    .dropna()
    .T
)

heatmap_data.index = ['']

# ------------------ Plot ------------------
fig, ax = plt.subplots(figsize=(6.5, 1.8))

sns.heatmap(
    heatmap_data,
    cmap='viridis_r',  # lower values appear lighter/yellower
    annot=True,
    fmt='.2f',
    annot_kws={'size': 10},
    linewidths=0.5,
    linecolor='white',
    cbar=False,
    ax=ax
)

# ------------------ Axis formatting ------------------
ax.set_yticks([])
ax.set_ylabel('')

ax.set_xlabel(
    'Deconvolution Tool',
    fontsize=11
)

ax.tick_params(
    axis='x',
    labelbottom=True,
    labeltop=False,
    bottom=True,
    top=False,
    labelsize=11,
    length=0,
    pad=8
)

plt.setp(
    ax.get_xticklabels(),
    rotation=30,
    ha='right'
)

# ------------------ Save ------------------
fig.savefig(
    'Heatmap_Random_JSD_TOO_DECODE_Optimal_Matrix_Stage3Val.svg',
    bbox_inches='tight'
)

plt.close(fig)